# 0. Imports

In [1]:
import numpy as np
import pandas as pd
import os 
import plotly.express as px

In [2]:
print(os.getcwd())
os.chdir("../")
os.getcwd()

c:\Users\leoco\Documents\cours\M2_MOSEF\ML_theory\land_value_prediction\notebooks


'c:\\Users\\leoco\\Documents\\cours\\M2_MOSEF\\ML_theory\\land_value_prediction'

In [3]:
train = pd.read_parquet("data/processed/train_test_out_sample_split/idf_vf_train.parquet")
train.head()

,date_mutation,nature_mutation,valeur_fonciere,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,code_type_local,...,commune_part_admin_sante,commune_etablissements_par_menage,commune_taux_etablissements_10_plus,commune_sante_score_2013_commune,commune_education_score_2013_commune,commune_revenu_score_2013_commune,commune_idh2_2013_commune,commune_taux_criminalite_moyen,commune_taux_croissance_pop,commune_densite_pop
0,2024-09-05,Vente,215000.0,RUE JEAN JACQUES ROUSSEAU,92150.0,92073,Suresnes,92,1,1.0,...,0.087753,0.071929,0.202652,0.694769,0.758865,0.665808,0.706481,0.421038,0.415631,1.305594e+10
1,2023-01-31,Vente,275000.0,RUE D AULNAY,93270.0,93071,Sevran,93,0,1.0,...,0.122034,0.050753,0.145763,0.621656,0.311248,0.231976,0.388294,0.593598,0.750298,7.106319e+09
2,2021-08-25,Vente,177885.0,RUE DES CHENES,92150.0,92073,Suresnes,92,1,2.0,...,0.087753,0.071929,0.202652,0.694769,0.758865,0.665808,0.706481,0.421038,0.102825,1.273456e+10
3,2021-03-12,Vente,124880.0,RUE DE LA CLOCHE,77300.0,77186,Fontainebleau,77,2,2.0,...,0.125677,0.115469,0.145179,0.691267,0.733403,0.568990,0.664553,0.564327,0.079089,8.652136e+07
4,2025-01-31,Vente,333100.0,RUE MIRIAM MAKEBA,93500.0,93055,Pantin,93,1,2.0,...,0.091778,0.101081,0.200765,0.590577,0.451268,0.222436,0.421427,0.767084,NaN,NaN


Supprimons les colonnes servant à construire la base mais non utiles pour la modélisation.  

In [4]:
def delete_useless_columns(df):
    
    columns_to_drop = [#valeur_fonciere, à garder pour d'éventuels analyses
                   #surface_reelle_bati, à garder pour d'éventuels analyses
                   "nature_mutation", 
                   #"adresse_nom_voie", on en a besoin pour retrouver lontitude lagitude manquantes 
                   "annee",
                   "semestre", 
                   "trimestre", 
                   "semestre_precedent", 
                   "annee_precedente", 
                   "trimestre_precedent"
                   ]

    return df.drop(columns=columns_to_drop).copy()

print(f"Shape initiale: {train.shape}")
train = delete_useless_columns(train)
train.shape

Shape initiale: (538199, 52)


(538199, 45)

# 1. Casting des variables 

In [5]:
train

,date_mutation,valeur_fonciere,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,code_type_local,type_local,...,commune_part_admin_sante,commune_etablissements_par_menage,commune_taux_etablissements_10_plus,commune_sante_score_2013_commune,commune_education_score_2013_commune,commune_revenu_score_2013_commune,commune_idh2_2013_commune,commune_taux_criminalite_moyen,commune_taux_croissance_pop,commune_densite_pop
0,2024-09-05,215000.0,RUE JEAN JACQUES ROUSSEAU,92150.0,92073,Suresnes,92,1,1.0,Maison,...,0.087753,0.071929,0.202652,0.694769,0.758865,0.665808,0.706481,0.421038,0.415631,1.305594e+10
1,2023-01-31,275000.0,RUE D AULNAY,93270.0,93071,Sevran,93,0,1.0,Maison,...,0.122034,0.050753,0.145763,0.621656,0.311248,0.231976,0.388294,0.593598,0.750298,7.106319e+09
2,2021-08-25,177885.0,RUE DES CHENES,92150.0,92073,Suresnes,92,1,2.0,Appartement,...,0.087753,0.071929,0.202652,0.694769,0.758865,0.665808,0.706481,0.421038,0.102825,1.273456e+10
3,2021-03-12,124880.0,RUE DE LA CLOCHE,77300.0,77186,Fontainebleau,77,2,2.0,Appartement,...,0.125677,0.115469,0.145179,0.691267,0.733403,0.568990,0.664553,0.564327,0.079089,8.652136e+07
4,2025-01-31,333100.0,RUE MIRIAM MAKEBA,93500.0,93055,Pantin,93,1,2.0,Appartement,...,0.091778,0.101081,0.200765,0.590577,0.451268,0.222436,0.421427,0.767084,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538194,2024-08-27,290000.0,AV DE BALZAC,91420.0,91432,Morangis,91,1,2.0,Appartement,...,0.045817,0.090105,0.260956,0.672979,0.439521,0.573113,0.561871,0.395804,0.756755,2.735625e+09
538195,2024-11-07,1234325.0,RUE CHASSELOUP LAUBAT,75015.0,75115,Paris 15e Arrondissement,75,1,2.0,Appartement,...,0.085000,0.100249,0.153647,0.722858,0.902589,0.715711,0.780386,0.694020,-0.467267,2.706038e+10
538196,2021-01-29,340000.0,AV DE PARIS,94800.0,94076,Villejuif,94,1,2.0,Appartement,...,0.115635,0.048360,0.153909,0.629372,0.507971,0.407285,0.514876,0.537509,-1.080173,1.025337e+10
538197,2022-09-26,384000.0,RUE DES POIRIERS BLANCS,77580.0,77142,Crécy-la-Chapelle,77,0,1.0,Maison,...,0.079096,0.085067,0.180791,0.593422,0.472925,0.575657,0.547335,0.292017,1.883932,2.958809e+08


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 538199 entries, 0 to 538198
Data columns (total 45 columns):
 #   Column                                           Non-Null Count   Dtype         
---  ------                                           --------------   -----         
 0   date_mutation                                    538199 non-null  datetime64[ns]
 1   valeur_fonciere                                  538199 non-null  float64       
 2   adresse_nom_voie                                 538198 non-null  object        
 3   code_postal                                      538198 non-null  float64       
 4   code_commune                                     538199 non-null  object        
 5   nom_commune                                      538199 non-null  object        
 6   code_departement                                 538199 non-null  object        
 7   nombre_lots                                      538199 non-null  int64         
 8   code_type_local         

In [7]:
def cast_variables(df):
    
    # Cast des colonnes spécifiques en category
    for col in ["code_postal", "code_type_local", "commune_revenu_median_2020"]:
        if col in df.columns:
            df[col] = df[col].astype('category')

    # Cast des colonnes object en category
    for col in df.select_dtypes(include='object').columns:
        if df[col].nunique() > 300:
            print(f"La colonne {col} a trop de modalités ({df[col].nunique()}) pour être castée en 'category'.")
            continue  # Skip cette colonne
        df[col] = df[col].astype('category')

    # Cast des colonnes numériques
    for col in df.select_dtypes(include=np.number).columns:
        # Vérifier si la colonne contient des décimales
        if df[col].dtype in ['float64', 'float32'] and not (df[col] % 1 == 0).all():
            df[col] = df[col].astype('float32')
            continue
            
        if df[col].min() >= 0:
            if df[col].max() <= 255:
                df[col] = df[col].astype('uint8')
            elif df[col].max() <= 65535:
                df[col] = df[col].astype('uint16')
            elif df[col].max() <= 4294967295:
                df[col] = df[col].astype('uint32')
            else:
                df[col] = df[col].astype('float32')
        else:
            if df[col].min() >= -128 and df[col].max() <= 127:
                df[col] = df[col].astype('int8')
            elif df[col].min() >= -32768 and df[col].max() <= 32767:
                df[col] = df[col].astype('int16')
            elif df[col].min() >= -2147483648 and df[col].max() <= 2147483647:
                df[col] = df[col].astype('int32')
            else:
                df[col] = df[col].astype('float32')
            
    return df.copy()

train = cast_variables(train)
train

La colonne adresse_nom_voie a trop de modalités (39301) pour être castée en 'category'.
La colonne code_commune a trop de modalités (1286) pour être castée en 'category'.
La colonne nom_commune a trop de modalités (1282) pour être castée en 'category'.


,date_mutation,valeur_fonciere,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,code_type_local,type_local,...,commune_part_admin_sante,commune_etablissements_par_menage,commune_taux_etablissements_10_plus,commune_sante_score_2013_commune,commune_education_score_2013_commune,commune_revenu_score_2013_commune,commune_idh2_2013_commune,commune_taux_criminalite_moyen,commune_taux_croissance_pop,commune_densite_pop
0,2024-09-05,215000.0,RUE JEAN JACQUES ROUSSEAU,92150.0,92073,Suresnes,92,1,1.0,Maison,...,0.087753,0.071929,0.202652,0.694769,0.758865,0.665808,0.706481,0.421038,0.415631,1.305594e+10
1,2023-01-31,275000.0,RUE D AULNAY,93270.0,93071,Sevran,93,0,1.0,Maison,...,0.122034,0.050753,0.145763,0.621656,0.311248,0.231976,0.388294,0.593598,0.750298,7.106319e+09
2,2021-08-25,177885.0,RUE DES CHENES,92150.0,92073,Suresnes,92,1,2.0,Appartement,...,0.087753,0.071929,0.202652,0.694769,0.758865,0.665808,0.706481,0.421038,0.102825,1.273456e+10
3,2021-03-12,124880.0,RUE DE LA CLOCHE,77300.0,77186,Fontainebleau,77,2,2.0,Appartement,...,0.125677,0.115469,0.145179,0.691267,0.733403,0.568990,0.664553,0.564327,0.079089,8.652136e+07
4,2025-01-31,333100.0,RUE MIRIAM MAKEBA,93500.0,93055,Pantin,93,1,2.0,Appartement,...,0.091778,0.101081,0.200765,0.590577,0.451268,0.222436,0.421427,0.767084,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538194,2024-08-27,290000.0,AV DE BALZAC,91420.0,91432,Morangis,91,1,2.0,Appartement,...,0.045817,0.090105,0.260956,0.672979,0.439521,0.573113,0.561871,0.395804,0.756755,2.735625e+09
538195,2024-11-07,1234325.0,RUE CHASSELOUP LAUBAT,75015.0,75115,Paris 15e Arrondissement,75,1,2.0,Appartement,...,0.085000,0.100249,0.153647,0.722858,0.902589,0.715711,0.780386,0.694020,-0.467267,2.706038e+10
538196,2021-01-29,340000.0,AV DE PARIS,94800.0,94076,Villejuif,94,1,2.0,Appartement,...,0.115635,0.048360,0.153909,0.629372,0.507971,0.407285,0.514876,0.537509,-1.080173,1.025337e+10
538197,2022-09-26,384000.0,RUE DES POIRIERS BLANCS,77580.0,77142,Crécy-la-Chapelle,77,0,1.0,Maison,...,0.079096,0.085067,0.180791,0.593422,0.472925,0.575657,0.547335,0.292017,1.883932,2.958809e+08


In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 538199 entries, 0 to 538198
Data columns (total 45 columns):
 #   Column                                           Non-Null Count   Dtype         
---  ------                                           --------------   -----         
 0   date_mutation                                    538199 non-null  datetime64[ns]
 1   valeur_fonciere                                  538199 non-null  float32       
 2   adresse_nom_voie                                 538198 non-null  object        
 3   code_postal                                      538198 non-null  category      
 4   code_commune                                     538199 non-null  object        
 5   nom_commune                                      538199 non-null  object        
 6   code_departement                                 538199 non-null  category      
 7   nombre_lots                                      538199 non-null  uint8         
 8   code_type_local         

# 2. Analyse des variables numériques  

#### Statistiques descriptives 

In [9]:
train.select_dtypes(include=np.number).describe()

,valeur_fonciere,nombre_lots,surface_reelle_bati,nombre_pieces_principales,surface_terrain,longitude,latitude,prix_m2,commune_prix_median_m2_semestre_prec,ecart_prix_median_pct,...,commune_part_admin_sante,commune_etablissements_par_menage,commune_taux_etablissements_10_plus,commune_sante_score_2013_commune,commune_education_score_2013_commune,commune_revenu_score_2013_commune,commune_idh2_2013_commune,commune_taux_criminalite_moyen,commune_taux_croissance_pop,commune_densite_pop
count,5.381990e+05,538199.000000,538199.000000,538199.000000,175120.000000,531283.000000,531283.000000,538199.000000,528702.000000,528702.000000,...,538199.000000,538199.000000,538199.000000,538197.000000,538199.000000,538132.000000,538132.000000,537436.000000,497987.000000,4.979870e+05
mean,3.939878e+05,1.086810,71.888892,3.227643,576.100708,2.360422,48.828705,6003.199707,5825.210449,7.211142,...,0.106322,0.092647,0.174461,0.646946,0.609935,0.548146,0.601684,0.575956,0.433812,9.693971e+09
std,3.524557e+05,0.968757,42.955576,1.550885,1805.761841,0.249955,0.144410,3653.826416,2981.649414,54.578922,...,0.043391,0.098430,0.057768,0.088436,0.200631,0.168450,0.139367,0.517506,1.194161,1.019753e+10
min,5.200000e+03,0.000000,10.000000,1.000000,1.000000,1.453111,48.125164,500.239990,729.927002,-96.338463,...,0.000000,0.011123,0.000000,0.000000,0.113469,0.000000,0.236949,0.000000,-9.889071,3.012685e+06
25%,1.950000e+05,0.000000,42.000000,2.000000,240.000000,2.248712,48.786373,3255.785767,3506.276611,-17.430599,...,0.079562,0.058081,0.138947,0.593166,0.459522,0.443765,0.507020,0.364954,-0.312900,2.020803e+09
50%,3.000000e+05,1.000000,65.000000,3.000000,401.000000,2.343398,48.853973,4764.706055,4480.500000,-1.184386,...,0.099314,0.072641,0.167832,0.650911,0.588408,0.563640,0.592777,0.457098,0.270653,5.725266e+09
75%,4.600000e+05,2.000000,90.000000,4.000000,550.000000,2.468547,48.904114,8214.286133,7760.984375,16.768448,...,0.122034,0.095523,0.206643,0.710094,0.786667,0.666696,0.713166,0.619372,0.914478,1.379592e+10
max,1.050000e+07,23.000000,880.000000,15.000000,448021.000000,3.518413,49.234760,19998.888672,19335.511719,998.010864,...,1.000000,1.299053,0.555556,1.000000,1.000000,0.947426,0.904551,12.795809,15.865758,3.995722e+10


#### Carte de visualisation des prix 

In [10]:
# Filtrer les données avec des coordonnées valides
train_map = train.dropna(subset=['longitude', 'latitude', 'prix_m2']).sample(frac=0.1, random_state=42)  # Échantillonner pour performance

# Créer la carte interactive avec plotly et une échelle de couleur plus progressive
fig = px.density_map(
    train_map, 
    lat='latitude', 
    lon='longitude', 
    z='prix_m2',
    radius=8,
    center=dict(lat=48.8566, lon=2.3522),
    zoom=8.5,
    # color_continuous_scale='Viridis',  
    range_color=[train_map['prix_m2'].quantile(0.05), train_map['prix_m2'].quantile(0.95)],  # Meilleure répartition
    labels={'prix_m2': 'Prix au m² (€)'},
    title='Carte interactive des prix au m² en Île-de-France',
    height=700
)

fig.update_traces(opacity=0.6)

fig.update_layout(
    mapbox_style="open-street-map",
    font=dict(size=12),
    title_font=dict(size=16, family='Arial Black'),
    coloraxis_colorbar=dict(title="Prix au m² (€)")
)

fig.show()

In [11]:
# Calculer le prix médian par commune
prix_par_commune = train.groupby(['code_commune', 'nom_commune']).agg({
    'prix_m2': 'median',
    'latitude': 'first',
    'longitude': 'first'
}).reset_index()

# Supprimer les valeurs manquantes
prix_par_commune = prix_par_commune.dropna(subset=['longitude', 'latitude', 'prix_m2'])

# Créer la carte avec agrégation par commune
# Créer des bins de 250 euros pour une échelle de couleur progressive
prix_min = prix_par_commune['prix_m2'].quantile(0.05)
prix_max = prix_par_commune['prix_m2'].quantile(0.95)
n_bins = int((prix_max - prix_min) / 250)

fig = px.scatter_map(
    prix_par_commune,
    lat='latitude',
    lon='longitude',
    color='prix_m2',
    size='prix_m2',
    hover_name='nom_commune',
    hover_data={'prix_m2': ':.0f', 'latitude': False, 'longitude': False},
    color_continuous_scale='Turbo',
    range_color=[prix_min, prix_max],
    color_continuous_midpoint=(prix_min + prix_max) / 2,
    size_max=20,
    zoom=8.5,
    center=dict(lat=48.8566, lon=2.3522),
    title='Prix médian au m² par commune en Île-de-France',
    labels={'prix_m2': 'Prix médian au m² (€)'},
    height=700
)

fig.update_layout(
    mapbox_style="open-street-map",
    font=dict(size=12),
    title_font=dict(size=16, family='Arial Black'),
    coloraxis_colorbar=dict(title="Prix médian au m² (€)")
)

fig.show()

#### Valeurs manquantes 

In [12]:
round(train.isnull().mean() * 100, 2)

date_mutation                                       0.00
valeur_fonciere                                     0.00
adresse_nom_voie                                    0.00
code_postal                                         0.00
code_commune                                        0.00
nom_commune                                         0.00
code_departement                                    0.00
nombre_lots                                         0.00
code_type_local                                     0.00
type_local                                          0.00
surface_reelle_bati                                 0.00
nombre_pieces_principales                           0.00
surface_terrain                                    67.46
longitude                                           1.29
latitude                                            1.29
prix_m2                                             0.00
commune_prix_median_m2_semestre_prec                1.76
ecart_prix_median_pct          

##### 1) Surface terrain 

In [13]:
train[train['surface_terrain'].isnull()]["code_postal"].value_counts(normalize=True).round(2) * 100

code_postal
75015.0    3.0
75018.0    3.0
75017.0    3.0
75011.0    2.0
75016.0    2.0
          ... 
77171.0    0.0
77157.0    0.0
77880.0    0.0
77134.0    0.0
77145.0    0.0
Name: proportion, Length: 526, dtype: float64

In [14]:
train[train['surface_terrain'] == 0].shape

(0, 45)

In [15]:
train["type_local"].value_counts(normalize=True).round(2) * 100

type_local
Appartement    68.0
Maison         32.0
Name: proportion, dtype: float64

In [16]:
train[train['surface_terrain'].isnull()]["type_local"].value_counts(normalize=True).round(2) * 100

type_local
Appartement    96.0
Maison          4.0
Name: proportion, dtype: float64

Si la surface terrain est manquante cela veut sûrement dire que le bien immobilier n'est pas associé à un terrain (ex: Paris intra muros).  On voit que ce sont très souvent des appartements. On va mettre à zéro les surfaces de terrain:

In [17]:
def fill_missing_surface_terrain(df):
    df['surface_terrain'] = df['surface_terrain'].fillna(0)
    return df.copy()

train = fill_missing_surface_terrain(train)
train

,date_mutation,valeur_fonciere,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,code_type_local,type_local,...,commune_part_admin_sante,commune_etablissements_par_menage,commune_taux_etablissements_10_plus,commune_sante_score_2013_commune,commune_education_score_2013_commune,commune_revenu_score_2013_commune,commune_idh2_2013_commune,commune_taux_criminalite_moyen,commune_taux_croissance_pop,commune_densite_pop
0,2024-09-05,215000.0,RUE JEAN JACQUES ROUSSEAU,92150.0,92073,Suresnes,92,1,1.0,Maison,...,0.087753,0.071929,0.202652,0.694769,0.758865,0.665808,0.706481,0.421038,0.415631,1.305594e+10
1,2023-01-31,275000.0,RUE D AULNAY,93270.0,93071,Sevran,93,0,1.0,Maison,...,0.122034,0.050753,0.145763,0.621656,0.311248,0.231976,0.388294,0.593598,0.750298,7.106319e+09
2,2021-08-25,177885.0,RUE DES CHENES,92150.0,92073,Suresnes,92,1,2.0,Appartement,...,0.087753,0.071929,0.202652,0.694769,0.758865,0.665808,0.706481,0.421038,0.102825,1.273456e+10
3,2021-03-12,124880.0,RUE DE LA CLOCHE,77300.0,77186,Fontainebleau,77,2,2.0,Appartement,...,0.125677,0.115469,0.145179,0.691267,0.733403,0.568990,0.664553,0.564327,0.079089,8.652136e+07
4,2025-01-31,333100.0,RUE MIRIAM MAKEBA,93500.0,93055,Pantin,93,1,2.0,Appartement,...,0.091778,0.101081,0.200765,0.590577,0.451268,0.222436,0.421427,0.767084,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538194,2024-08-27,290000.0,AV DE BALZAC,91420.0,91432,Morangis,91,1,2.0,Appartement,...,0.045817,0.090105,0.260956,0.672979,0.439521,0.573113,0.561871,0.395804,0.756755,2.735625e+09
538195,2024-11-07,1234325.0,RUE CHASSELOUP LAUBAT,75015.0,75115,Paris 15e Arrondissement,75,1,2.0,Appartement,...,0.085000,0.100249,0.153647,0.722858,0.902589,0.715711,0.780386,0.694020,-0.467267,2.706038e+10
538196,2021-01-29,340000.0,AV DE PARIS,94800.0,94076,Villejuif,94,1,2.0,Appartement,...,0.115635,0.048360,0.153909,0.629372,0.507971,0.407285,0.514876,0.537509,-1.080173,1.025337e+10
538197,2022-09-26,384000.0,RUE DES POIRIERS BLANCS,77580.0,77142,Crécy-la-Chapelle,77,0,1.0,Maison,...,0.079096,0.085067,0.180791,0.593422,0.472925,0.575657,0.547335,0.292017,1.883932,2.958809e+08


##### 2) Retrouver un proxy de lattitude et longitude 

[todo] Rajouter le code voie pour retrouver les coordonnées 

In [18]:
train[train['latitude'].isnull()]

,date_mutation,valeur_fonciere,adresse_nom_voie,code_postal,code_commune,nom_commune,code_departement,nombre_lots,code_type_local,type_local,...,commune_part_admin_sante,commune_etablissements_par_menage,commune_taux_etablissements_10_plus,commune_sante_score_2013_commune,commune_education_score_2013_commune,commune_revenu_score_2013_commune,commune_idh2_2013_commune,commune_taux_criminalite_moyen,commune_taux_croissance_pop,commune_densite_pop
45,2023-01-20,190000.0,RUE JULES CHATENAY,93380.0,93059,Pierrefitte-sur-Seine,93,1,2.0,Appartement,...,0.072340,0.064359,0.124823,0.591285,0.252541,0.141425,0.328417,NaN,1.359640,9.050147e+09
80,2022-07-04,490000.0,RUE DE BORDEAUX,95400.0,95019,Arnouville,95,0,1.0,Maison,...,0.083004,0.101840,0.110672,0.648876,0.306902,0.392695,0.449491,0.536531,0.291456,5.045423e+09
171,2022-06-28,350000.0,AV ROGER SALENGRO,92370.0,92022,Chaville,92,1,2.0,Appartement,...,0.109223,0.045074,0.123786,0.676168,0.775818,0.697982,0.716656,0.327765,0.979751,5.831549e+09
430,2022-04-15,494950.0,CHE DU CLOS DU ROI,91530.0,91630,Le Val-Saint-Germain,91,0,1.0,Maison,...,0.138889,0.059603,0.027778,0.618993,0.632776,0.686656,0.646142,0.094102,0.894843,1.186953e+08
641,2023-09-27,470000.0,AV DES MYOSOTIS,93370.0,93047,Montfermeil,93,0,1.0,Maison,...,0.093516,0.084418,0.132170,0.587574,0.303692,0.357874,0.416380,0.408910,1.235079,5.124404e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537921,2021-06-30,590000.0,ALL SUZANNE,92160.0,92002,Antony,92,0,1.0,Maison,...,0.126415,0.058081,0.204403,0.722403,0.768743,0.681758,0.724302,0.431882,0.339688,6.544979e+09
537933,2021-02-19,166000.0,ALL DES PEUPLIERS,95350.0,95539,Saint-Brice-sous-Forêt,95,2,2.0,Appartement,...,0.085919,0.073382,0.186158,0.816811,0.545884,0.509164,0.623953,0.476843,0.840453,2.480500e+09
538039,2021-01-13,190000.0,RUE GABRIELLE JOSSERAND,93500.0,93055,Pantin,93,2,2.0,Appartement,...,0.091778,0.101081,0.200765,0.590577,0.451268,0.222436,0.421427,0.767084,1.824759,1.147345e+10
538057,2023-06-28,100000.0,RES LE PARC DE PETIT BOURG,91000.0,91228,Évry-Courcouronnes,91,2,2.0,Appartement,...,0.148098,0.085168,0.237136,0.591242,0.519623,0.264171,0.458346,0.516017,3.868148,5.263858e+09


#### Valeurs aberrantes

#### Corrélations

# 3. Analyse des varaibles catégorielles 